In [1]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, roc_auc_score, classification_report)

SEED = 11
OUT = "compas_assignment_outputs"
os.makedirs(OUT, exist_ok=True)
URLS = [
    "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv",
    "https://github.com/propublica/compas-analysis/raw/master/compas-scores-two-years.csv",
]

# -------------------- Load and prepare dataset --------------------
def load_data():
    last_error = None
    for url in URLS:
        try:
            df = pd.read_csv(url)
            print(f"Loaded dataset from: {url}")
            return df
        except Exception as e:
            last_error = e
    raise RuntimeError("Could not download COMPAS dataset. Download compas-scores-two-years.csv from "
                       "the ProPublica compas-analysis GitHub repository and place it beside this script. "
                       f"Last error: {last_error}")

df_raw = load_data()
print("Raw shape:", df_raw.shape)
print("Available columns:", list(df_raw.columns))

# Assignment calls the binary target 'is_recidivist'. In this commonly used COMPAS file,
# 'two_year_recid' indicates recidivism within two years; use it as the binary outcome.
# decile_score is Model A's continuous target.
required = ["decile_score", "two_year_recid", "age", "sex", "race", "priors_count", "c_charge_degree"]
missing_cols = [c for c in required if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Dataset is missing expected columns: {missing_cols}")
df = df_raw.copy()
df["outcome"] = pd.to_numeric(df["two_year_recid"], errors="coerce")
df["decile_score"] = pd.to_numeric(df["decile_score"], errors="coerce")
# Restrict to usable target rows. Predictors are imputed within pipelines.
df = df[df["outcome"].isin([0, 1]) & df["decile_score"].notna()].copy()
df["outcome"] = df["outcome"].astype(int)
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["priors_count"] = pd.to_numeric(df["priors_count"], errors="coerce")
df["sex"] = df["sex"].fillna("Unknown").astype(str)
df["race"] = df["race"].fillna("Unknown").astype(str)
df["c_charge_degree"] = df["c_charge_degree"].fillna("Unknown").astype(str)

# Standard feature set used for the predictive models. Sensitive race is included for auditing;
# this is a classroom exercise, not a deployment recommendation.
NUM = ["age", "priors_count"]
CAT = ["sex", "race", "c_charge_degree"]
FEATURES = NUM + CAT
X = df[FEATURES]
y_linear = df["decile_score"]
y_logistic = df["outcome"]
X_train, X_test, yA_train, yA_test, yB_train, yB_test, idx_train, idx_test = train_test_split(
    X, y_linear, y_logistic, df.index, test_size=0.30, random_state=SEED, stratify=y_logistic
)

numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
try:
    encoder = OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)
except TypeError:  # compatibility with older scikit-learn
    encoder = OneHotEncoder(handle_unknown="ignore", drop="first", sparse=False)
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", encoder)])
preprocessor = ColumnTransformer([("num", numeric_pipe, NUM), ("cat", cat_pipe, CAT)])

# -------------------- Part 1: build and compare models --------------------
modelA = Pipeline([("prep", preprocessor), ("model", LinearRegression())])
modelB = Pipeline([("prep", preprocessor), ("model", LogisticRegression(max_iter=3000, class_weight=None))])
modelA.fit(X_train, yA_train)
modelB.fit(X_train, yB_train)
A_pred = modelA.predict(X_test)
B_prob = modelB.predict_proba(X_test)[:, 1]
A_median = float(np.median(A_pred))
A_class = (A_pred >= A_median).astype(int)
B_class = (B_prob >= 0.5).astype(int)
cm = confusion_matrix(A_class, B_class, labels=[0, 1])
mismatch = int(np.sum(A_class != B_class))
mismatch_pct = 100 * mismatch / len(A_class)
print("\n=== PART 1: MODEL COMPARISON ===")
print(f"Rows used: {len(df)} | Train: {len(X_train)} | Test: {len(X_test)} | random_state={SEED}")
print(f"Model A median predicted decile_score threshold: {A_median:.4f}")
print(f"Model B probability threshold: 0.5000")
print("Confusion matrix (rows=Model A low/high, columns=Model B low/high):\n", cm)
print(f"Mismatches: {mismatch}/{len(A_class)} = {mismatch_pct:.2f}%")
print(f"Model B test accuracy: {accuracy_score(yB_test, B_class):.4f}")
try: print(f"Model B test ROC-AUC: {roc_auc_score(yB_test, B_prob):.4f}")
except ValueError: pass
pd.DataFrame({"actual_recidivism": np.asarray(yB_test), "linear_decile_prediction": A_pred,
              "linear_high_by_test_median": A_class, "logistic_probability": B_prob,
              "logistic_high_at_0.5": B_class}, index=idx_test).to_csv(f"{OUT}/part1_test_predictions.csv", index_label="row_index")
pd.DataFrame(cm, index=["A_low", "A_high"], columns=["B_low", "B_high"]).to_csv(f"{OUT}/part1_confusion_matrix.csv")

# Helper for statsmodels logistic regressions; imputes, encodes, and standardizes consistently.
def design_matrix(frame, columns, fit_columns=None):
    temp = frame[columns].copy()
    num_cols = [c for c in columns if c in NUM]
    cat_cols = [c for c in columns if c in CAT]
    for c in num_cols:
        temp[c] = pd.to_numeric(temp[c], errors="coerce")
        temp[c] = temp[c].fillna(temp[c].median())
        sd = temp[c].std(ddof=0)
        temp[c] = (temp[c] - temp[c].mean()) / (sd if sd and np.isfinite(sd) else 1)
    for c in cat_cols:
        temp[c] = temp[c].fillna("Unknown").astype(str)
    temp = pd.get_dummies(temp, columns=cat_cols, drop_first=True, dtype=float)
    temp = temp.astype(float)
    if fit_columns is not None:
        temp = temp.reindex(columns=fit_columns, fill_value=0.0)
    return sm.add_constant(temp, has_constant="add")

def fit_sm_logit(frame, target="outcome", columns=None):
    columns = columns or FEATURES
    Xsm = design_matrix(frame, columns)
    y = frame[target].astype(int)
    return sm.Logit(y, Xsm).fit(disp=False, maxiter=200)

def coef_table(result, model_name):
    ci = result.conf_int()
    return pd.DataFrame({"model": model_name, "coefficient": result.params,
                         "ci_low": ci[0], "ci_high": ci[1], "p_value": result.pvalues})

# -------------------- Part 2.1: Simpson's paradox / subgroup effect --------------------
print("\n=== PART 2.1: SUBGROUP ANALYSIS (race) ===")
full_res = fit_sm_logit(df)
# Compare full model to two large race groups; only fit groups with both outcome classes.
race_counts = df["race"].value_counts()
major_races = [r for r in ["African-American", "Caucasian"] if race_counts.get(r, 0) >= 50]
if len(major_races) < 2:
    major_races = list(race_counts[race_counts >= 100].index[:2])
sub_results = {"Full dataset": full_res}
for race_name in major_races:
    sub = df[df["race"] == race_name]
    if sub["outcome"].nunique() == 2 and len(sub) > 30:
        try: sub_results[race_name] = fit_sm_logit(sub)
        except Exception as e: print(f"Could not fit subgroup {race_name}: {e}")
coef_frames = []
for name, res in sub_results.items():
    tab = coef_table(res, name)
    tab["term"] = tab.index
    coef_frames.append(tab.reset_index(drop=True))
subcoef = pd.concat(coef_frames, ignore_index=True)
subcoef.to_csv(f"{OUT}/part2_1_subgroup_coefficients.csv", index=False)
key_terms = [t for t in subcoef["term"].unique() if t == "priors_count"]
if key_terms:
    print(subcoef[subcoef.term == "priors_count"].to_string(index=False))
else:
    print("priors_count coefficient not available; inspect saved subgroup coefficient table.")
print("Race groups fitted:", list(sub_results.keys()))

# -------------------- Part 2.2: omitted-variable bias --------------------
print("\n=== PART 2.2: OMITTED-VARIABLE BIAS (drop race) ===")
full_ovb = fit_sm_logit(df, columns=FEATURES)
reduced_features = [c for c in FEATURES if c != "race"]
reduced_ovb = fit_sm_logit(df, columns=reduced_features)
full_tab = coef_table(full_ovb, "Full")
reduced_tab = coef_table(reduced_ovb, "Reduced_without_race")
# Compare same named predictors, including dummy levels shared by both specifications.
ovb = full_tab[["coefficient", "ci_low", "ci_high", "p_value"]].add_prefix("full_").join(
    reduced_tab[["coefficient", "ci_low", "ci_high", "p_value"]].add_prefix("reduced_"), how="inner")
ovb["absolute_change"] = (ovb["reduced_coefficient"] - ovb["full_coefficient"]).abs()
ovb["percent_change"] = np.where(ovb["full_coefficient"].abs() > 1e-10,
    100 * (ovb["reduced_coefficient"] - ovb["full_coefficient"]) / ovb["full_coefficient"].abs(), np.nan)
ovb.index.name = "predictor"
ovb.to_csv(f"{OUT}/part2_2_omitted_variable_bias.csv")
print("Dropped confounder: race")
print(ovb.head(12).round(4).to_string())

# -------------------- Part 2.3: adversarial subset --------------------
print("\n=== PART 2.3: HIGH-CONFIDENCE WRONG PREDICTIONS ===")
actual = np.asarray(yB_test).astype(int)
wrong = ((B_prob >= 0.75) & (actual == 0)) | ((B_prob <= 0.25) & (actual == 1))
adv = X_test.copy()
adv["actual_outcome"] = actual
adv["predicted_probability"] = B_prob
adv["predicted_class"] = B_class
adv["high_confidence_wrong"] = wrong
adv["error_type"] = np.where((B_prob >= .75) & (actual == 0), "False positive",
                       np.where((B_prob <= .25) & (actual == 1), "False negative", ""))
wrong_rows = adv[adv.high_confidence_wrong].copy()
wrong_count = len(wrong_rows)
wrong_pct = 100 * wrong_count / len(adv)
if wrong_count > 50:
    wrong_rows = wrong_rows.sample(n=50, random_state=SEED)
wrong_rows.to_csv(f"{OUT}/part2_3_adversarial_subset.csv", index_label="row_index")
print(f"All high-confidence errors in test set: {wrong_count}/{len(adv)} = {wrong_pct:.2f}%")
print(f"Rows in analyzed adversarial subset: {len(wrong_rows)}")
print("Full test gender distribution (%):\n", adv.sex.value_counts(normalize=True).mul(100).round(2))
print("Adversarial subset gender distribution (%):\n", wrong_rows.sex.value_counts(normalize=True).mul(100).round(2) if len(wrong_rows) else "No rows")
print("Full test race distribution (%):\n", adv.race.value_counts(normalize=True).mul(100).round(2))
print("Adversarial subset race distribution (%):\n", wrong_rows.race.value_counts(normalize=True).mul(100).round(2) if len(wrong_rows) else "No rows")
if len(wrong_rows):
    print("Age summary, full test:\n", adv.age.describe().round(2))
    print("Age summary, adversarial subset:\n", wrong_rows.age.describe().round(2))
    print("Prior-count summary, adversarial subset:\n", wrong_rows.priors_count.describe().round(2))
    print("Top categorical values in adversarial subset:")
    for col in ["sex", "race", "c_charge_degree"]:
        print(col, wrong_rows[col].value_counts(normalize=True).head(5).mul(100).round(1).to_dict())

# -------------------- Part 2.4: fairness threshold sweep --------------------
print("\n=== PART 2.4: FAIRNESS TRADE-OFFS ===")
# Compare African-American and Caucasian groups, the two principal groups in COMPAS analysis.
# If either group is absent from test set, choose the two most common test-set race categories.
race_test = np.asarray(X_test["race"].astype(str))
A_name, B_name = "African-American", "Caucasian"
if not ((race_test == A_name).any() and (race_test == B_name).any()):
    vals = pd.Series(race_test).value_counts().index.tolist()
    if len(vals) < 2: raise ValueError("Need at least two race groups for fairness analysis")
    A_name, B_name = vals[0], vals[1]

def safe_ratio(a, b):
    # Undefined if denominator is zero; retain NaN instead of pretending fairness is perfect.
    return float(a / b) if b != 0 else np.nan

def group_rates(y, pred, mask):
    yy, pp = y[mask], pred[mask]
    positive_rate = np.mean(pp == 1) if len(pp) else np.nan
    pos_actual = yy == 1
    neg_actual = yy == 0
    tpr = np.mean(pp[pos_actual] == 1) if pos_actual.sum() else np.nan
    fpr = np.mean(pp[neg_actual] == 1) if neg_actual.sum() else np.nan
    ppv = np.mean(yy[pp == 1] == 1) if (pp == 1).sum() else np.nan
    return positive_rate, tpr, fpr, ppv

threshold_rows = []
for t in np.round(np.arange(0.05, 0.951, 0.1), 2):
    pred = (B_prob >= t).astype(int)
    maskA, maskB = race_test == A_name, race_test == B_name
    dpA, tprA, fprA, ppvA = group_rates(actual, pred, maskA)
    dpB, tprB, fprB, ppvB = group_rates(actual, pred, maskB)
    dp = safe_ratio(dpA, dpB)
    tpr_ratio, fpr_ratio = safe_ratio(tprA, tprB), safe_ratio(fprA, fprB)
    # Assignment defines EO as min(TPR_A/TPR_B, FPR_A/FPR_B).
    eo = np.nanmin([tpr_ratio, fpr_ratio]) if not (np.isnan(tpr_ratio) and np.isnan(fpr_ratio)) else np.nan
    pp = safe_ratio(ppvA, ppvB)
    threshold_rows.append({"threshold": t, "DP_ratio_A_over_B": dp, "EO_min_TPR_FPR_ratio": eo,
                           "PP_ratio_A_over_B": pp, "DP_A": dpA, "DP_B": dpB,
                           "TPR_A": tprA, "TPR_B": tprB, "FPR_A": fprA, "FPR_B": fprB,
                           "PPV_A": ppvA, "PPV_B": ppvB,
                           "accuracy_overall": accuracy_score(actual, pred)})
fair = pd.DataFrame(threshold_rows)
fair.to_csv(f"{OUT}/part2_4_fairness_thresholds.csv", index=False)
print(f"Group A = {A_name}; Group B = {B_name}")
print(fair[["threshold", "DP_ratio_A_over_B", "EO_min_TPR_FPR_ratio", "PP_ratio_A_over_B", "accuracy_overall"]].round(4).to_string(index=False))

# Trade-off chart: axes as required, points labelled with threshold.
fig, ax = plt.subplots(figsize=(8, 6))
valid = fair.dropna(subset=["DP_ratio_A_over_B", "EO_min_TPR_FPR_ratio"])
ax.scatter(valid["DP_ratio_A_over_B"], valid["EO_min_TPR_FPR_ratio"])
for _, r in valid.iterrows():
    ax.annotate(f"t={r['threshold']:.2f}", (r["DP_ratio_A_over_B"], r["EO_min_TPR_FPR_ratio"]),
                xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_xlim(0, 1.5); ax.set_ylim(0, 1.5)
ax.set_xlabel("Demographic Parity ratio (Group A / Group B)")
ax.set_ylabel("Equalized Odds ratio (assignment definition)")
ax.set_title("COMPAS fairness trade-off across thresholds")
ax.grid(True, alpha=.3)
fig.tight_layout(); fig.savefig(f"{OUT}/part2_4_fairness_tradeoff.png", dpi=200); plt.close(fig)

# Optional plot for probability distribution and errors.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(B_prob[actual == 0], bins=20, alpha=.6, label="Actual outcome 0")
ax.hist(B_prob[actual == 1], bins=20, alpha=.6, label="Actual outcome 1")
ax.axvline(.25, linestyle="--"); ax.axvline(.5, linestyle="--"); ax.axvline(.75, linestyle="--")
ax.set_xlabel("Predicted probability of recidivism"); ax.set_ylabel("Count")
ax.set_title("Logistic model probabilities on test set"); ax.legend()
fig.tight_layout(); fig.savefig(f"{OUT}/test_probability_distribution.png", dpi=200); plt.close(fig)

# -------------------- Preprocessing documentation --------------------
preprocessing_notes = f"""PREPROCESSING NOTES (edit only if your actual run differs)
Dataset: ProPublica COMPAS two-year recidivism file.
Rows after requiring valid decile_score and binary two_year_recid: {len(df)}.
Missing values: target rows missing decile_score/two_year_recid are excluded. Numeric predictors
(age, priors_count) are median-imputed inside the predictive pipeline. Categorical predictors
(sex, race, c_charge_degree) are mode-imputed and one-hot encoded with drop_first=True.
Feature scaling: numeric predictors are standardized inside the model pipeline. The separate
statsmodels coefficient audits standardize numeric columns and one-hot encode categorical columns.
Outliers: no rows are automatically removed based on outlier thresholds; this preserves unusual
cases for the assignment's adversarial analysis. Report this explicitly and discuss its limitation.
Split: 70% train / 30% test, random_state={SEED}, stratified by the binary outcome.
Outcome mapping: the supplied file uses two_year_recid (recidivism within two years), not a column
literally named is_recidivist; this is used as the binary logistic target.
"""
with open(f"{OUT}/preprocessing_notes.txt", "w", encoding="utf-8") as f: f.write(preprocessing_notes)

# Short numerical summary for the report. The eight essays and 500-word reflection should be
# written only after inspecting these actual results; do not submit generic answers without values.
summary = {
    "seed": SEED, "rows_used": len(df), "train_rows": len(X_train), "test_rows": len(X_test),
    "linear_prediction_median": A_median, "mismatch_count": mismatch,
    "mismatch_percent": mismatch_pct, "test_modelB_accuracy_at_0.5": accuracy_score(yB_test, B_class),
    "high_confidence_wrong_count": wrong_count, "high_confidence_wrong_percent_of_test": wrong_pct,
    "fairness_group_A": A_name, "fairness_group_B": B_name,
}
pd.Series(summary, name="value").to_csv(f"{OUT}/summary_numbers.csv", header=True)
print("\n=== SUMMARY ===")
for k, v in summary.items(): print(f"{k}: {v}")
print(f"\nAll outputs saved in: {os.path.abspath(OUT)}")
print("Important: inspect CSV tables and chart, then use those actual numbers in the written report.")


Loaded dataset from: https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv
Raw shape: (7214, 53)
Available columns: ['id', 'name', 'first', 'last', 'compas_screening_date', 'sex', 'dob', 'age', 'age_cat', 'race', 'juv_fel_count', 'decile_score', 'juv_misd_count', 'juv_other_count', 'priors_count', 'days_b_screening_arrest', 'c_jail_in', 'c_jail_out', 'c_case_number', 'c_offense_date', 'c_arrest_date', 'c_days_from_compas', 'c_charge_degree', 'c_charge_desc', 'is_recid', 'r_case_number', 'r_charge_degree', 'r_days_from_arrest', 'r_offense_date', 'r_charge_desc', 'r_jail_in', 'r_jail_out', 'violent_recid', 'is_violent_recid', 'vr_case_number', 'vr_charge_degree', 'vr_offense_date', 'vr_charge_desc', 'type_of_assessment', 'decile_score.1', 'score_text', 'screening_date', 'v_type_of_assessment', 'v_decile_score', 'v_score_text', 'v_screening_date', 'in_custody', 'out_custody', 'priors_count.1', 'start', 'end', 'event', 'two_year_recid']

=== PART 1

In [ ]:
# -------------------- Extra numbers cited in the PDF report (added cell) --------------------
from scipy.stats import spearmanr, norm
print("\n=== EXTRA: DATA FACTS (Preprocessing section) ===")
print("Null counts in used columns:", df_raw[["decile_score","two_year_recid","age","sex","race","priors_count","c_charge_degree"]].isna().sum().to_dict())
print("Reoffenders overall:", int(df.outcome.sum()), f"({100*df.outcome.mean():.2f}%) | train {int(yB_train.sum())} ({100*yB_train.mean():.2f}%) | test {int(yB_test.sum())} ({100*yB_test.mean():.2f}%)")
print("Race counts:", df.race.value_counts().to_dict())
print("Age / priors summary:\n", df[["age","priors_count","decile_score"]].describe().round(2))
print("priors_count skew:", round(df.priors_count.skew(),2), "| defendants with >=10 priors:", int((df.priors_count>=10).sum()))
print("Race base rates:", df.groupby("race").outcome.mean().round(4).to_dict())
print("Test AA / Caucasian counts:", int((X_test.race=="African-American").sum()), int((X_test.race=="Caucasian").sum()))

print("\n=== EXTRA: PART 1 (precision / recall / agreement detail) ===")
from sklearn.metrics import precision_score, recall_score
yt = np.asarray(yB_test).astype(int)
for name, c in [("Model A median rule", A_class), ("Model B at 0.5", B_class)]:
    print(f"{name}: flagged {int(c.sum())} ({100*c.mean():.2f}%) acc {accuracy_score(yt,c):.4f} prec {precision_score(yt,c):.4f} rec {recall_score(yt,c):.4f}")
print("Model A prediction range:", round(A_pred.min(),2), "to", round(A_pred.max(),2), "| rescaled median (m-1)/9:", round((A_median-1)/9,3))
print("Spearman(A_pred, B_prob):", round(spearmanr(A_pred, B_prob)[0],3))
m1 = (A_class==1)&(B_class==0); m2 = (A_class==0)&(B_class==1)
print("A high / B low:", int(m1.sum()), "mean logistic P", round(B_prob[m1].mean(),3), "range", round(B_prob[m1].min(),3), round(B_prob[m1].max(),3))
print("A low / B high:", int(m2.sum()), "mean logistic P", round(B_prob[m2].mean(),3))
print("Flag-nobody accuracy:", round(1-yt.mean(),4))

print("\n=== EXTRA: PART 2.1 (Wald tests on subgroup differences) ===")
for t in ["priors_count","sex_Male","age","c_charge_degree_M"]:
    a = subcoef[(subcoef.model=="African-American")&(subcoef.term==t)].iloc[0]
    c = subcoef[(subcoef.model=="Caucasian")&(subcoef.term==t)].iloc[0]
    sa = (a.ci_high-a.ci_low)/(2*1.959964); sc = (c.ci_high-c.ci_low)/(2*1.959964)
    d = a.coefficient-c.coefficient; se = np.hypot(sa,sc); z = d/se
    print(f"{t}: AA {a.coefficient:.4f} Cauc {c.coefficient:.4f} diff {d:.4f} z {z:.2f} p {2*(1-norm.cdf(abs(z))):.4f} | OR AA {np.exp(a.coefficient):.3f} Cauc {np.exp(c.coefficient):.3f}")

print("\n=== EXTRA: PART 2.3 (all 115 errors, subset breakdown, calibration of tails) ===")
fp = (B_prob>=.75)&(yt==0); fn = (B_prob<=.25)&(yt==1)
print("All errors: FP", int(fp.sum()), "FN", int(fn.sum()))
print(f"P>=0.75 wrong: {fp.sum()}/{(B_prob>=.75).sum()} = {fp.sum()/(B_prob>=.75).sum():.3f} | P<=0.25 wrong: {fn.sum()}/{(B_prob<=.25).sum()} = {fn.sum()/(B_prob<=.25).sum():.3f}")
sub = wrong_rows.copy()
print("Subset error types:", sub.error_type.value_counts().to_dict())
print(sub.groupby("error_type")[["age","priors_count","predicted_probability"]].agg(["mean","min","max"]).round(2))
print("Subset share age>=40:", (sub.age>=40).mean(), "| priors<=3:", (sub.priors_count<=3).mean(), "| priors>=9:", (sub.priors_count>=9).mean())
print("Full test share age>=40:", round((X_test.age>=40).mean(),4), "| priors<=3:", round((X_test.priors_count<=3).mean(),4), "| priors>=9:", round((X_test.priors_count>=9).mean(),4))
print("Test priors summary:", X_test.priors_count.describe().round(2).to_dict())
print("Test charge degree shares:", X_test.c_charge_degree.value_counts(normalize=True).round(4).to_dict())
print(pd.crosstab(sub.error_type, sub.race))

print("\n=== EXTRA: PART 2.4 (group rates, trade-off arithmetic) ===")
print(fair[["threshold","DP_A","DP_B","TPR_A","TPR_B","FPR_A","FPR_B","PPV_A","PPV_B"]].round(4).to_string(index=False))
f2 = fair.set_index("threshold")
def shift(t0, t1):
    out = []
    for k in ["DP_ratio_A_over_B","EO_min_TPR_FPR_ratio","PP_ratio_A_over_B"]:
        out.append(f"{k.split('_')[0]} {f2.loc[t0,k]:.4f}->{f2.loc[t1,k]:.4f} ({100*(f2.loc[t1,k]-f2.loc[t0,k])/f2.loc[t0,k]:+.1f}%)")
    print(f"t {t0} -> {t1}: " + "; ".join(out) + f"; accuracy {f2.loc[t0,'accuracy_overall']:.4f}->{f2.loc[t1,'accuracy_overall']:.4f}")
shift(0.55, 0.35); shift(0.35, 0.65)
print("FPR ratio by threshold:", (f2.FPR_A/f2.FPR_B).round(3).to_dict())
print("Max gap from 1.0 by threshold:", f2[["DP_ratio_A_over_B","EO_min_TPR_FPR_ratio","PP_ratio_A_over_B"]].sub(1).abs().max(axis=1).round(4).to_dict())
